In [4]:
import psutil

In [7]:
!du -sh /Users/charlottel/Library/Caches/cython/brian_extensions

 18M	/Users/charlottel/Library/Caches/cython/brian_extensions


In [9]:
mem = psutil.virtual_memory()
print(f"总内存: {mem.total / 1e9:.1f} GB")
print(f"已用: {mem.used/1e9:.1f} GB ({mem.percent:.1f}%)")
print(f"可用: {mem.available/1e9:.1f} GB")
print(f"Swap: {psutil.swap_memory().used / 1e9:.1f} GB")

总内存: 17.2 GB
已用: 6.0 GB (57.8%)
可用: 7.3 GB
Swap: 0.0 GB


# 完整操作步骤（内存优化版）

## 准备阶段（10分钟）

### Step 1: 创建新模块文件

```bash
# Terminal或notebook
mkdir -p flylif/utils
touch flylif/utils/__init__.py
```

### Step 2: 保存新文件

保存以下3个文件（复制对应artifact）：

1. **flylif/core/simulation.py**（替换旧版）
   - 来源：Artifact "simulation.py"
   - 修改：添加了gc.collect()清理

2. **flylif/utils/checkpoint.py**（新建）
   - 来源：Artifact "checkpoint.py"
   
3. **flylif/utils/memory_utils.py**（新建）
   - 来源：Artifact "memory_utils.py"

### Step 3: 更新__init__.py

**flylif/utils/__init__.py**（可选）：
```python
from .checkpoint import CheckpointManager
from .memory_utils import print_memory, check_memory_safe, MemoryMonitor

__all__ = ['CheckpointManager', 'print_memory', 'check_memory_safe', 'MemoryMonitor']
```

---

## 修改notebook（15分钟）

### Step 4: 打开exp1_clean_test.ipynb

### Step 5: 添加BATCH_CONFIG定义

**在Cell 4（参数设置）后面添加：**
```python
# Batch configuration (for memory management)
BATCH_CONFIG = {
    'batch1': list(range(10, 80, 10)),   # 7 frequencies
    'batch2': list(range(80, 150, 10)),  # 7 frequencies
    'batch3': list(range(150, 210, 10)), # 5 frequencies
}
```

### Step 6: 替换Section 4（并行执行）

**删除原来的Cell 6-7，替换为：**
- Cell 6A: 导入checkpoint和监控（来自artifact "exp1_batched_cells"）
- Cell 6B: Worker函数（带清理）
- Cell 6C: run_batch()函数
- Cell 7A: Batch 1执行
- Cell 7B: Batch 2执行
- Cell 7C: Batch 3执行
- Cell 8: 合并结果

**共7个新cells替换原来2个**

---

## 执行阶段（60-70分钟）

### Session 1: Batch 1（20-25分钟）

```
1. Restart Kernel（清空内存）
2. Run Cells 1-6C（setup + 函数定义，约30秒）
3. Run Cell 7A（Batch 1）
   - 执行中：观察内存输出
   - 预期：20-25分钟
   - 完成后：查看内存状态
4. 结果保存在：checkpoints/exp1/
```

### Session 2: Batch 2（20-25分钟）

```
5. Kernel → Restart Kernel（清空内存）
6. Run Cells 1-6C（重新setup）
7. Run Cell 7B（Batch 2）
   - Checkpoint自动跳过已完成的batch1
   - 只运行batch2的7个频率
8. 完成后查看内存
```

### Session 3: Batch 3（15-20分钟）

```
9. Kernel → Restart Kernel
10. Run Cells 1-6C
11. Run Cell 7C（Batch 3）
    - 只运行最后5个频率
12. Run Cell 8（Merge）
    - 从checkpoint加载全部19个频率
    - 获得完整results_dict
```

### Session 4: 分析对比（5分钟）

```
13. 继续运行原来的Section 5-7
    - Cell 9-13（结果处理、对比、保存）
    - results_dict已包含全部19频率数据
```

---

## 预期输出

### Batch 1运行时

```
======================================================================
Batch: Batch 1
======================================================================

Configuration:
  Frequencies: [10, 20, 30, 40, 50, 60, 70]
  Trials/freq: 10
  Workers: 3
  Checkpoint: ./checkpoints/exp1

  Memory status: ✓ Memory OK

  Running 7 tasks...

[Batch 1] Start - Memory: 55.3%

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done   1 tasks      | elapsed:  9.2min
[Parallel(n_jobs=3)]: Done   3 tasks      | elapsed: 10.1min
[Parallel(n_jobs=3)]: Done   5 tasks      | elapsed: 18.4min
[Parallel(n_jobs=3)]: Done   7 out of   7 | elapsed: 21.3min finished

[Batch 1] End - Memory: 68.2% (Δ+2.1GB)

  ✅ Batch complete in 21.30 min
  Final memory: Memory: 68.2% used, 5.4GB free, Swap: 0.0GB

======================================================================
✅ Batch 1 complete!
======================================================================

⚠️ NEXT STEP:
1. Check memory status above
2. Kernel → Restart Kernel
3. Re-run Cells 1-6C (setup)
4. Run Cell 7B (Batch 2)
```

---

## 故障恢复

### 如果中途中断（例如在Batch 2第3个任务）

```
1. 不要慌，checkpoint已保存：
   - Batch 1全部（7个）
   - Batch 2部分（2个）

2. Restart Kernel

3. Re-run Cells 1-6C

4. Re-run Cell 7B
   - Checkpoint自动识别：已完成batch1全部+batch2前2个
   - 只运行剩余5个

5. 继续
```

---

## 内存监控要点

### 正常输出

```
每个worker完成后：
[task_10Hz] End - Memory: 62.1% (Δ+1.2GB)  ← 正常
[task_20Hz] End - Memory: 64.3% (Δ+1.1GB)  ← 正常
...
```

### 异常警告

```
[task_50Hz] End - Memory: 88.5% (Δ+3.5GB)
  ⚠️ Memory warning: 88.5%  ← 危险信号

此时应该：
1. 观察下一个任务
2. 如果继续升高 → 中断
3. Restart kernel → 继续checkpoint恢复
```

---

## 最终输出（Merge后）

```
======================================================================
Merging All Batches
======================================================================

✅ Loaded 19 frequencies

Complete Results:
   Frequency  Active Neurons  Total Spikes
          10              32           234
          20              87          1234
         ...             ...           ...
         200             356         29876

✅ All 19 frequencies complete!
   Continue to Section 5 (Results Processing)
```

---

## 时间估算

| 阶段 | 时间 | 累计 |
|------|------|------|
| 准备文件 | 10 min | 10 min |
| 修改notebook | 15 min | 25 min |
| **Batch 1** | 20-25 min | 50 min |
| Restart+setup | 1 min | 51 min |
| **Batch 2** | 20-25 min | 76 min |
| Restart+setup | 1 min | 77 min |
| **Batch 3** | 15-20 min | 97 min |
| Merge+分析 | 5 min | 102 min |

**总计：约1.7小时（包含3次restart）**

---

## Checklist

**准备：**
- [ ] 创建flylif/utils/目录
- [ ] 保存simulation.py（更新版）
- [ ] 保存checkpoint.py
- [ ] 保存memory_utils.py
- [ ] 修改exp1_clean_test.ipynb

**执行：**
- [ ] Session 1: Batch 1（20min）
- [ ] Session 2: Batch 2（20min）
- [ ] Session 3: Batch 3（15min）
- [ ] Merge并分析

**验证：**
- [ ] 19频率全部完成
- [ ] 与原文r > 0.85
- [ ] 内存未超过80%
- [ ] 无swap使用

---

## 成功标准

✅ 全部19频率×10 trials完成  
✅ 每批内存<75%，无swap  
✅ Checkpoint机制验证（可中断恢复）  
✅ 与原文相关性>0.85  
✅ 总时间<2小时

# 操作清单V1：可视化与分析模块

**前置条件**：Exp1已完成（20频率×10trials，checkpoint已保存）

**目标**：
1. 创建可复用的可视化工具
2. 与原论文详细对比分析
3. 生成publication-quality图表
4. 为Exp2/3准备分析模板

---

## Part 1: 创建可视化模块（10分钟）

### Step 1: 保存visualization.py

**文件位置**: `flylif/utils/visualization.py`

**来源**: Artifact "visualization.py"（包含4个函数）
- `plot_correlation()` - 相关性散点图
- `plot_response_heatmap()` - 响应热图
- `plot_frequency_response_curve()` - 频率响应曲线
- `plot_summary_statistics()` - 汇总统计条形图

**验证**: 
```python
from flylif.utils.visualization import plot_correlation
# 成功导入即可
```

---

## Part 2: Notebook分析部分（10分钟）

### Step 2: 修改exp1_clean_test.ipynb

**当前状态**：Cell 8已完成merge，有results_dict和df_summary

**修改Section 4（分析部分）**：

**删除**：原Cell 8-12（如果有旧的results processing）

**替换为**：Artifact "exp1_analysis_cells"的4个cells
- Cell 9: 计算firing rates
- Cell 10: 加载原文数据，计算相关性
- Cell 11: 生成4个可视化图
- Cell 12: 保存完整结果

**位置**：在Cell 8（Merge）之后插入

---

## Part 3: 执行分析（5分钟）

### Step 3: 运行分析cells

**前提**：Batch 1-4已完成，checkpoint有20个频率数据

**操作**：
1. 确认Cell 8（Merge）已运行，有results_dict
2. 依次运行Cell 9-12
3. 观察输出

**预期输出**：
```
Section 4: Results Analysis
======================================================================

[1/4] Extracting firing rates...
   ✅ Processed 20 frequencies

[2/4] Summary statistics...
   Frequency  Active Neurons  Total Spikes
          10              36          2437
         ...             ...           ...
         200             388        166009

======================================================================
[3/4] Comparison with Shiu et al. (2024)
======================================================================

100Hz Comparison:
  Original (30 trials): 404 active neurons
  Ours (10 trials):     323 active neurons
  Pearson r: 0.85XX
  ✅ Excellent (r > 0.85)

  Top 50 overlap: 42/50 (84%)

======================================================================
[4/4] Visualization
======================================================================

Plot 1: Correlation scatter
   → correlation_100Hz.png

Plot 2: Response heatmap
   Showing top 323 neurons
   → response_heatmap.png

Plot 3: MN9 frequency response
   → mn9_response_curve.png

Plot 4: Summary statistics
   → summary_stats.png

✅ All plots saved to: ./results/exp1_full

======================================================================
✅ Experiment 1 Complete!
======================================================================

Key results:
  Frequencies tested: 20
  Total active neurons: XXX
  Correlation with paper: r = 0.8XXX

Output: ./results/exp1_full/
```

---

## Part 4: 验证与保存（5分钟）

### Step 4: 检查生成的文件

```python
# 新cell
import os
from pathlib import Path

output_dir = Path('./results/exp1_full')

print("生成的文件:")
print("\nData files:")
for f in ['exp1_complete_results.pkl', 'summary.csv', 'firing_rate_matrix.csv']:
    path = output_dir / f
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"  ✓ {f:30s} ({size:.1f} KB)")

print("\nPlots:")
for f in output_dir.glob('*.png'):
    print(f"  ✓ {f.name}")
```

### Step 5: 查看图片质量

**在Jupyter或Finder中打开**：
- `correlation_100Hz.png` - 检查散点分布
- `response_heatmap.png` - 检查热图清晰度
- `mn9_response_curve.png` - 检查曲线趋势
- `summary_stats.png` - 检查统计图

---

## Part 5: 提取可复用分析模板（10分钟）

### Step 6: 创建analysis_utils.py（可选）

**文件位置**: `flylif/utils/analysis_utils.py`

**内容**：
```python
"""
Analysis utilities for experiment results.
Functions for calculating statistics, correlations, etc.
"""

def calculate_firing_rates(results_dict, duration_s=1.0, n_trials=10):
    """Extract firing rates from results."""
    all_firing_rates = {}
    for freq, res in results_dict.items():
        df = res['df']
        if len(df) > 0:
            counts = df.groupby('flywire_id').size()
            rates = counts / (n_trials * duration_s)
            all_firing_rates[freq] = rates.to_dict()
        else:
            all_firing_rates[freq] = {}
    return all_firing_rates

def compare_with_original(rate_ours, original_parquet_path, 
                         n_trials_original=30):
    """Compare with original paper results."""
    # ... 加载和对比逻辑
    return correlation, overlap_percent

# 等函数
```

**用途**：Exp2/3可直接调用，无需重写计算逻辑

---

## 后续实验准备（参考）

### Exp2小规模测试notebook

```
exp2_sufficiency_test.ipynb:

Section 1-3: Setup（复用exp1的imports和data loading）
Section 4: Small-scale test（20神经元×2频率×3T）
Section 5: Analysis
├─ 使用plot_response_heatmap()
└─ 不同：Y轴是测试的神经元，X轴是频率
Section 6: Save

预期：10分钟运行 + 可视化
```

### Exp3小规模测试notebook

```
exp3_silencing_test.ipynb:

Section 1-3: Setup
Section 4: Small-scale test（10神经元×2频率×3T）
Section 5: Analysis  
├─ 使用plot_heatmap()
└─ Y轴=沉默的神经元，颜色=相对MN9活性%
Section 6: Save

预期：15分钟运行 + 可视化
```

---

## Checklist

**文件保存：**
- [ ] visualization.py（必须）
- [ ] analysis_utils.py（可选，建议有）

**Notebook修改：**
- [ ] 删除旧Cell 8-9（如有）
- [ ] 插入新Cell 9-12（分析可视化）

**执行分析：**
- [ ] Run Cell 9（firing rates）
- [ ] Run Cell 10（对比原文）
- [ ] Run Cell 11（4个图）
- [ ] Run Cell 12（保存）

**验证输出：**
- [ ] r值记录（期望>0.85）
- [ ] 4个图生成
- [ ] firing_rate_matrix.csv存在
- [ ] 图片质量检查

**文档更新：**
- [ ] PROGRESS.md添加今日成果
- [ ] Git commit（可选）

---

## 时间估算

```
文件准备：10分钟
Notebook修改：10分钟
运行分析：5分钟
验证结果：5分钟

总计：30分钟
```

---

## 下一步

**今天下午计划**：
1. 完成Exp1分析可视化（30分钟）
2. 创建exp2_test.ipynb小规模测试（30分钟创建+10分钟运行）
3. 创建exp3_test.ipynb小规模测试（30分钟创建+15分钟运行）

**今天产出**：
- ✅ Exp1完整数据+分析+图表
- ✅ Exp2/3逻辑验证
- ✅ 可视化工具模块
- → 明天可直接云端部署

# Day 2 Push操作清单

## 准备阶段（5分钟）

### Step 1: 整理notebook imports

**操作**：
1. 打开exp1_clean_test.ipynb
2. 删除Section 1的所有旧import cells
3. 替换为Artifact "organized_imports"的单个cell（Section 1.1-1.7）
4. 保存

**验证**：重新运行Section 1，应无报错

---

### Step 2: 清空notebook输出

**操作**：
```
Jupyter菜单：
Cell → All Output → Clear
保存
```

**重要**：避免push大量输出数据到GitHub

---

### Step 3: 追加PROGRESS.md

**操作**：
1. 打开PROGRESS.md
2. 在文件末尾追加Artifact "progress_update_day2"的内容
3. 保存

---

## Git提交（5分钟）

### Step 4: 检查文件状态

```bash
cd /Users/charlottel/MyLibrary/connectome/LIFmodel/LIF_simulation

git status
```

**应该看到修改的文件**：
```
modified:   flylif/core/simulation.py
modified:   exp1_clean_test.ipynb
modified:   PROGRESS.md

新文件：
flylif/utils/checkpoint.py
flylif/utils/memory_utils.py
flylif/utils/visualization.py
```

---

### Step 5: 添加文件

```bash
# 新模块
git add flylif/utils/checkpoint.py
git add flylif/utils/memory_utils.py
git add flylif/utils/visualization.py
git add flylif/utils/__init__.py

# 更新的文件
git add flylif/core/simulation.py
git add exp1_clean_test.ipynb
git add PROGRESS.md

# 检查
git status
```

---

### Step 6: 提交

```bash
git commit -m "Day 2: Exp1 complete + memory optimization

Features:
- Checkpoint system (incremental save/resume)
- Memory monitoring (prevent swap)
- Visualization tools (4 reusable functions)
- Efficient ID conversion (244 IDs, r=0.93)

Performance:
- 3-worker batched execution (68 min)
- Memory stable (<65%, swap <1GB)
- Full Exp1 data (20×10T)

Validation:
- r = 0.9263 vs Shiu et al.
- ID conversion improved +11%
- Top 50 overlap 86%
"
```

---

### Step 7: Push

```bash
git push origin main
```

---

## 可选：创建Tag

```bash
git tag -a v0.2.0 -m "Exp1 Complete

- 20 freqs × 10 trials
- r = 0.9263 correlation  
- Memory optimized
- Checkpoint system
"

git push origin v0.2.0
```

---

## 检查结果（2分钟）

### Step 8: 验证GitHub

**访问你的GitHub仓库**，检查：
- [ ] 新commits可见
- [ ] 3个新文件在flylif/utils/
- [ ] PROGRESS.md更新
- [ ] exp1_clean_test.ipynb无输出（文件较小）

---

## Checklist

**准备：**
- [ ] 整理notebook imports（Section 1）
- [ ] 清空notebook输出
- [ ] 追加PROGRESS.md

**Git操作：**
- [ ] git add（6个文件）
- [ ] git commit
- [ ] git push
- [ ] (可选) git tag v0.2.0

**验证：**
- [ ] GitHub可见更新
- [ ] 文件完整
- [ ] Commit message清晰

---

## 预计时间

```
整理imports：3分钟
清空输出：1分钟
追加文档：1分钟
Git操作：5分钟

总计：10分钟
```

---

## 今日成果摘要

```
代码：
✅ 3个新模块（checkpoint, memory, visualization）
✅ 1个更新（simulation.py内存清理）

数据：
✅ Exp1完整（20×10T）
✅ r = 0.9263验证
✅ 4个publication图

系统：
✅ 内存问题解决
✅ Checkpoint机制work
✅ 可复用可视化工具

下一步：
→ Exp2/3小规模测试（明天）
```

# Exp2/3实现 + 云端部署

## 背景（100字）

FlyLIF：Drosophila脑LIF模型复现（Shiu et al. 2024）

**已完成**：
- 模块化（8个.py）
- 数据优化（68MB）
- Exp1验证（r=0.93，20×10T）
- 内存优化（3核分批，checkpoint）

**本次目标**：
- 实现Exp2充分性测试（200神经元×8频率）
- 实现Exp3沉默实验（200×8，含权重快照优化）
- 准备48核云端部署

---

## 核心架构（50字）

```
flylif/
├── core/ (parameters, data_loader, network, simulation)
└── utils/ (checkpoint, memory, visualization, cave_utils)

关键优化：
- 数据预处理（68MB）
- 3核分批+checkpoint
- gc.collect()防泄漏
```

---

## Exp2/3参数

**Exp2**（figures.ipynb Fig 1e）：
- 200神经元（top_200_neurons.npy）逐个激活
- 8频率×10T = 1600次仿真
- 目标：测哪些能激活MN9

**Exp3**（Fig 1f）：
- 21 Sugar GRNs持续激活
- 200神经元逐个沉默
- 8频率×10T = 1600次仿真
- 关键：syn.w[indices]=0可能慢（需权重快照优化）

---

## 今天任务

**小规模测试**（2小时）：
1. Exp2：20神经元×2频率×3T（验证逻辑）
2. Exp3：10神经元×2频率×3T（测试沉默+优化）

**如果成功** → 准备云端脚本
**如果问题** → 调试优化

---

## 附件说明

**原论文**：figures.ipynb, model.py（参考实现）
**我们的**：8个.py模块, exp1_clean_test.ipynb（模板）
**文档**：PROGRESS.md（优化历程）

---

## 提问示例

**启动对话**：
"开始Exp2小规模测试，20神经元×2频率×3trials"

**遇到问题**：
"Exp3沉默实验慢（160秒/trial），讨论权重快照优化"

**准备云端**：
"创建run_exp2_cloud.py，48核部署"

---

## 关键约束

- 16GB内存 → 3核稳定
- Checkpoint必须 → 长时间运行
- 简短讨论模式 → 省token（除非要求全面）

1. figures.ipynb - 5个实验的完整代码
2. model.py - 核心run_exp函数+并行逻辑
3. utils.py - 结果处理工具

用途：参考Exp2/3的原文实现


flylif/core/
├── parameters.py      ← 必须
├── data_loader.py     ← 必须（68MB优化）
├── network.py         ← 必须
└── simulation.py      ← 必须（含gc清理）

flylif/utils/
├── checkpoint.py      ← 必须（Exp2/3长时间运行需要）
├── memory_utils.py    ← 必须（内存监控）
├── visualization.py   ← 必须（画图复用）
└── cave_utils.py      ← 必须（ID转换）

1. PROGRESS.md - 工作日志（了解优化历程）
2. README.md - 项目概览（快速理解）

# Exp2/3 实施指南

**日期**: 2026-01-27  
**状态**: 小规模测试阶段  
**目标**: 验证batch优化后再跑完整实验

---

## 🚀 快速开始（5分钟上手）

### 前置条件检查

```bash
# 1. 检查Exp1输出
ls results/exp1_full/top_200_neurons.npy  # 必须存在

# 2. 激活环境
conda activate flylif

# 3. 测试新模块
python -c "from flylif.core.experiments import run_sufficiency_freq_batch; print('✅')"
```

### 立即运行测试

**Exp2测试** (20神经元×2频率×3T，预计3-5分钟)：
```bash
jupyter notebook notebooks/exp2_test_clean.ipynb
```

**执行**：
1. 从`exp1_clean_test.ipynb`复制Cells 1-4（环境setup）
2. 运行Section 5-8（测试+验证）
3. 检查总时间：应在3-5分钟内

**Exp3测试** (10神经元×2频率×3T，预计5-7分钟)：
```bash
jupyter notebook notebooks/exp3_test_clean.ipynb
```

---

## 📊 技术背景（为什么这么做）

### 性能瓶颈来源

**你之前的测试数据对比**：

| 实验 | 配置 | 总时间 | 单次仿真 | 问题 |
|------|------|--------|---------|------|
| Exp1优化后 | 21neu×5freq×10T | **68 min** (4批) | 56秒 | ✅ 正常 |
| Exp2未优化 | 20neu×2freq×1T | **96 min** | 144秒 | ❌ 慢3× |
| Exp3未优化 | 20neu×2freq×1T | **134 min** | 191秒 | ❌ 慢4× |

**元凶**：每个神经元都重建网络
```python
# 慢版本（未优化）
for neuron in top_200:  # 200次循环
    NET = build_network(...)  # 每次7秒 → 23分钟浪费！
    run_simulation(NET, [neuron], ...)

# 快版本（优化后）
NET = build_network(...)  # 只build 1次
for neuron in batch_25:   # 25次循环，共享网络
    net.restore('initial')  # 只restore，<0.1秒
    run_simulation(NET, [neuron], ...)
```

### 关键优化

1. **Batch处理**：25个神经元共享1个网络
2. **只保存summary**：省95%空间（800MB→40MB）
3. **Frequency级checkpoint**：8个频率，易断点续传
4. **保持3 workers**：内存安全（<70%，swap<1GB）

---

## 📁 文件说明

### 新增文件

```
flylif/core/
└── experiments.py              ← 核心批量函数

notebooks/
├── exp2_test_clean.ipynb       ← Exp2小规模测试（20×2×3T）
└── exp3_test_clean.ipynb       ← Exp3小规模测试（10×2×3T）

HOWTO_EXP23.md                  ← 本文档
```

### 与exp1的关系

**复用exp1的setup**：
- Cells 1-4: 环境、FlyWire、模块导入、配置
- 这些cell在exp2/3中**标记为"COPY from exp1"**
- 原因：节省token，这些cell内容完全一样

**Exp2/3特有部分**：
- Section 5+: 实验逻辑（sufficiency/necessity）
- 新增`experiments.py`模块调用

---

## 🔬 详细操作流程

### Exp2测试（充分性）

#### **目标**
测试哪些神经元**能够激活MN9**（单独刺激时）

#### **步骤**

**1. 打开notebook**
```bash
jupyter notebook notebooks/exp2_test_clean.ipynb
```

**2. Section 1-4：Setup（从exp1复制）**

按照每个Section的markdown说明：
```markdown
## 📋 **Copy from exp1_clean_test.ipynb**

Instructions:
1. Open exp1_clean_test.ipynb
2. Copy cells and paste below
```

**或者更快的方法**：
```python
# 先运行exp1_clean_test.ipynb的Cells 1-4
# 然后在同一个kernel里打开exp2_test_clean.ipynb
# 这样DATA、NET已经存在，可以跳过setup
```

**3. Section 5：配置**
```python
# 这个cell会加载top_200_neurons.npy
# 并设置测试参数：20神经元, 2频率, 3 trials
```

**4. Section 6：运行测试**
```python
# 两个cell分别测试50Hz和100Hz
# 每个cell运行时间：~1.5-2.5分钟
```

**预期输出**：
```
Running Sufficiency Test - Frequency 50 Hz
  Sufficiency batch @ 50 Hz (20 neurons)
    Progress: 10/20 (85s, ETA: 85s)    ← 进度更新
    Progress: 20/20 (170s, ETA: 0s)
    ✅ Complete (170.3s)
```

**5. Section 7：验证结果**
```python
# 自动显示summary statistics
# 检查有多少神经元能激活MN9
# Top 5神经元排名
```

**6. Section 8：保存**
- 输出：`./results/exp2_test/sufficiency_test_results.pkl`

---

### Exp3测试（必要性）

#### **目标**
测试哪些神经元**是MN9激活所必需的**（沉默后MN9下降>20%）

#### **步骤**

**1-3. Setup**（同Exp2，从exp1复制）

**4. Section 4：配置**
```python
# 关键差异：
NEURONS_SILENCE = top_200[:10]   # 要沉默的神经元
NEU_ACTIVATE = NEU_SUGAR_LEFT    # 21个Sugar GRNs持续激活
```

**5. Section 5：运行测试**
```python
# 每个频率包含：
# - 1次Control（无沉默，baseline）
# - 10次Silencing（逐个沉默）
```

**预期输出**：
```
Running Necessity Test - Frequency 50 Hz
  Necessity batch @ 50 Hz (10 neurons)
    [1/2] Control (no silencing)...
       Control MN9: 45.3 ± 5.2 Hz      ← baseline
    [2/2] Silencing tests...
       Progress: 10/10 (210s, ETA: 0s)
    ✅ Complete (225.1s)
```

**6. Section 6：分析结果**
```python
# 自动识别required neurons
# Threshold: relative_% < 80%
# 示例：
#   Neuron X: 30 Hz (66%) ← Required
#   Neuron Y: 50 Hz (110%) ← Not required
```

---

## ⏱️ 性能预期

### 小规模测试（统一条件对比）

| 实验 | 配置 | Baseline投影 | 优化后预计 | 加速比 |
|------|------|-------------|-----------|--------|
| **Exp2** | 20×2×**1T** | **9.6分钟** (96.2×20/200) | **2-3分钟** | **4×** |
| **Exp3** | 10×2×**1T** | **67分钟** (134×10/20) | **3-4分钟** | **17×** |

**注意**：统一Trial数=1，才能与baseline对比

### 全尺度投影（成功后）

**本地** (3 workers, 16GB RAM)：
| 实验 | 配置 | 时间/Batch | Batches | 总时间 |
|------|------|-----------|---------|--------|
| Exp2 | 200×8×10T | ~30分钟 | 4批×2频率 | **~2小时** |
| Exp3 | 200×8×10T | ~30分钟 | 4批×2频率 | **~2小时** |

**云端** (48 workers, 96GB RAM)：
- Exp2/3各自：**~5-10分钟**
- 成本：~$0.10 (Spot实例)

---

## 🐛 常见问题

### 问题1：找不到top_200_neurons.npy

**症状**：
```python
FileNotFoundError: results/exp1_full/top_200_neurons.npy not found
```

**解决**：
```bash
# 运行exp1完整流程
jupyter notebook notebooks/exp1_clean_test.ipynb
# 运行到Section 6（保存Top 200 neurons）
```

---

### 问题2：NET不存在

**症状**：
```python
NameError: name 'NET' is not defined
```

**解决**：
```python
# 方案A：先运行exp1_clean_test.ipynb的Cells 1-4（推荐）
# 方案B：在当前notebook build网络
from flylif.core.network import build_network
NET = build_network(data=DATA, ...)  # 复制exp1的build_network调用
```

---

### 问题3：测试超时（>10分钟）

**可能原因**：
1. **内存swap** → Restart kernel，运行`print_memory()`检查
2. **重复build网络** → 检查verbose输出，不应出现"Building network"
3. **网络规模**（不太可能，同样的DATA）

**诊断**：
```python
# 在测试前
print_memory("Before test: ")

# 在测试中观察verbose输出
# 不应该看到 "[1/7] Creating neurons" 这种build步骤
# 应该只看到 "Progress: X/20"
```

---

### 问题4：所有神经元MN9都是0 Hz

**检查target neuron**：
```python
print(f"Target: {TARGET_MN9}")
print(f"In network: {TARGET_MN9[0] in NET['flyid2i']}")

# 对照Exp1结果
# 100Hz Sugar GRNs → MN9应该是40-60 Hz
```

---

### 问题5：Exp3所有relative都是100%

**检查control**：
```python
# Control应该>0
print(f"Control: {result_50hz['control']}")
# 如果是0，说明21 Sugar GRNs没激活MN9（异常）

# 检查沉默是否work
# verbose输出应有 "Silenced synapses: XXX"
```

---

## 📈 成功后的下一步

### 1. 更新PROGRESS.md

```markdown
## 2026-01-27: Exp2/3 Batch优化测试

### 完成
- **批量函数实现** (`experiments.py`)
  - `run_sufficiency_freq_batch()`: 共享网络，单频率处理25神经元
  - `run_necessity_freq_batch()`: 包含control baseline
  
- **小规模验证**
  - Exp2测试：20×2×3T，X分钟（vs 96分钟baseline）
  - Exp3测试：10×2×3T，Y分钟（vs 134分钟baseline）
  - 加速：~20×

### 性能对比
| 指标 | 未优化 | 优化后 | 改善 |
|------|--------|--------|------|
| Exp2测试 | 96分钟 | X分钟 | XX× |
| 单次build开销 | 200×7秒 | 1×7秒 | 200× |

### 下一步
1. 创建exp2_full.ipynb（200×8×10T）
2. 4-batch执行（每批2频率，~30分钟）
3. 同样流程完成Exp3
```

---

### 2. Git提交

```bash
git add flylif/core/experiments.py
git add notebooks/exp2_test_clean.ipynb
git add notebooks/exp3_test_clean.ipynb
git add HOWTO_EXP23.md
git add PROGRESS.md

git commit -m "Add Exp2/3 batch implementation with 20× speedup

- Implement run_sufficiency_freq_batch() and run_necessity_freq_batch()
- Optimize: shared network across neurons (1 build vs 200 builds)
- Add small-scale test notebooks (20neu, 10neu)
- Expected full-scale: 2h local or 5min cloud (vs 38h naive)"

git push
```

---

### 3. 创建full版本

```bash
# 复制测试notebook
cp notebooks/exp2_test_clean.ipynb notebooks/exp2_full.ipynb
cp notebooks/exp3_test_clean.ipynb notebooks/exp3_full.ipynb
```

**修改参数**（在exp2_full.ipynb）：
```python
# === Section 5: Configuration ===

# Full-scale parameters
NEURONS_FULL = top_200_neurons  # All 200 (was [:20])
FREQS_FULL = [25, 50, 75, 100, 125, 150, 175, 200]  # 8 freqs (was [50, 100])
N_TRIALS_FULL = 10  # 10 trials (was 3)

# Batch configuration (similar to Exp1)
BATCH_CONFIG = {
    'batch1': [25, 50],     # 2 frequencies per batch
    'batch2': [75, 100],
    'batch3': [125, 150],
    'batch4': [175, 200],
}
```

**添加checkpoint**（参考exp1_clean_test.ipynb）：
```python
from flylif.utils.checkpoint import CheckpointManager

ckpt = CheckpointManager('./checkpoints/exp2')

# Run batch with checkpoint
for batch_name, freqs in BATCH_CONFIG.items():
    print(f"\n{'='*70}")
    print(f"Batch: {batch_name}")
    print(f"{'='*70}")
    
    for freq in freqs:
        if ckpt.is_completed(freq):
            print(f"  ✅ {freq} Hz already completed")
            continue
        
        result = run_sufficiency_freq_batch(
            NET, top_200_neurons, TARGET_MN9, 
            freq=freq, n_trials=10
        )
        
        ckpt.save(freq, result)
    
    print(f"\n⚠️  Batch {batch_name} complete")
    print(f"   Restart kernel before next batch")
```

---

## 🔧 高级选项

### save_full_data接口

**默认**（推荐）：
```python
result = run_sufficiency_freq_batch(..., save_full_data=False)
# 只保存 {'mean': 45.2, 'std': 3.1}
```

**如果需要完整spike数据**：
```python
result = run_sufficiency_freq_batch(..., save_full_data=True)
# 额外保存 result['raw_data'][neuron_id] = DataFrame(t, trial, flywire_id)
# 用途：自定义分析、画raster plot、检查trial一致性
```

---

### Batch size调整

**当前默认**：25神经元/batch

**如果内存紧张**：
```python
# 在experiments.py中手动分批
neurons_per_batch = 10  # 减少到10
batches = [top_200[i:i+10] for i in range(0, 200, 10)]
```

**如果内存充足**（云端96GB）：
```python
neurons_per_batch = 50  # 增加到50
# 减少任务数，提高效率
```

---

## 📋 完整操作清单

### 阶段1：测试验证（今天）

- [ ] 保存5个文件到对应位置
- [ ] 打开exp2_test_clean.ipynb
- [ ] 复制exp1的setup cells（Sections 1-4）
- [ ] 运行Exp2测试（Section 5-8）
- [ ] 记录实际时间：______ 分钟
- [ ] 检查内存峰值：______ %
- [ ] 验证结果格式正确
- [ ] 打开exp3_test_clean.ipynb
- [ ] 复制setup cells
- [ ] 运行Exp3测试
- [ ] 记录时间：______ 分钟
- [ ] 更新PROGRESS.md
- [ ] Git commit + push

**成功标准**：
- ✅ Exp2: 3-5分钟（vs 96分钟）
- ✅ Exp3: 5-7分钟（vs 134分钟）
- ✅ 内存<70%，swap<1GB
- ✅ 结果格式正确

---

### 阶段2：全尺度本地（明天或晚些）

- [ ] 创建exp2_full.ipynb（copy from test）
- [ ] 修改参数：200神经元，8频率，10 trials
- [ ] 添加4-batch结构（参考exp1）
- [ ] 添加checkpoint机制
- [ ] 运行Batch 1（2频率）
- [ ] Restart kernel
- [ ] 运行Batch 2-4
- [ ] 总时间验证：~2小时
- [ ] 重复Exp3

---

### 阶段3：云端部署（可选）

- [ ] 准备云端脚本（调整workers=48）
- [ ] 测试8-core实例（验证）
- [ ] 部署48-core实例
- [ ] 运行Exp2+Exp3（~10分钟）
- [ ] 下载结果

---

## 🎯 关键决策记录

### Q1: 理解Exp1时间 ✅
- Batch4的20.9分钟是5个频率的总时间
- 平均4.2分钟/频率（21个Sugar GRNs）
- Exp2单神经元预计更快（2-3分钟/频率）

### Q2: 只保存firing rate ✅
- **省空间**：95%（800MB→40MB）
- **省时间**：<5%（主要是I/O）
- **决定**：做成接口（`save_full_data=False`默认）
- **原因**：灵活性，防止未来需要完整数据

### Q3: Batch与Restart ✅
- **保持**：8频率分4批，每批后restart
- **原因**：防止内存累积，即使有gc.collect()
- **参考**：Exp1的4-batch成功经验

### Q4: 测试→云端流程 ✅
- **流程**：小规模baseline → 优化验证 → full本地 → 云端
- **云端调整**：workers 3→48，batch 4→1-2
- **其他不变**：数据处理、核心逻辑

### Q5: Notebook vs Script ✅
- **选择**：Notebook（bash环境问题）
- **优势**：调试方便，cell选择性运行，内存可视化
- **复用**：从exp1复制setup cells（节省token）

---

## 📞 需要帮助时

**如果测试失败**：
1. 记录完整错误信息
2. 记录实际runtime和内存峰值
3. 检查verbose输出（有没有unexpected的"Building network"）
4. 提供结果的snapshot（前3个神经元的数据）

**如果测试成功**：
1. 汇报实际时间（对比预期）
2. 确认要不要继续full版本
3. 讨论batch size优化（25 vs 10 vs 50）

---

**版本**: 0.2.0-alpha  
**作者**: Rui Luo  
**最后更新**: 2026-01-27